# Predicted vs Actual
Regression scatter plot for solubility.


In [5]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

MODEL_DIR = 'xgboost'
MODEL_NAME = 'XGBoost'

cwd = Path.cwd()
project_root = Path("..").resolve() if cwd.name == MODEL_DIR else cwd
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from qspr_config import (
    DATA_PATH,
    ECFP_RADIUS,
    ECFP_N_BITS,
    N_ESTIMATORS,
    N_ESTIMATORS_GRID,
    TOP_N_BITS,
    RANDOM_SEED,
    TEST_SIZE,
    CV_FOLDS,
    N_JOBS,
    FIG_DPI,
    FIGSIZE_SQUARE,
    FIGSIZE_SQUARE,
)
from qspr_common import (
    load_dataset,
    build_feature_matrix,
    make_binary_target,
    apply_plot_style,
    resolve_output_dir,
)

apply_plot_style()
out_dir = resolve_output_dir(MODEL_DIR)


In [6]:
df = load_dataset(DATA_PATH)
df, X = build_feature_matrix(df, radius=ECFP_RADIUS, n_bits=ECFP_N_BITS)


[22:56:49] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:50] WARNING: not removing hydrogen atom without neighbors
[22:56:51] WARNING: not removing hydrogen atom without neighbors
[22:56:52] WARNING: not removing hydrogen atom without neighbors
[22:56:53] WARNING: not removing hydrogen atom without neighbors
[22:56:53] WARNING: not removing hydrogen atom without neighbors
[22:56:53] WARNING: not r

In [7]:
y = df["Solubility"].to_numpy()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)

model = XGBRegressor(
    n_estimators=N_ESTIMATORS,
    random_state=RANDOM_SEED,
    n_jobs=N_JOBS,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    verbosity=0,
)
model.fit(x_train, y_train)
y_pred = model.predict(x_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

fig, ax = plt.subplots(figsize=FIGSIZE_SQUARE)
ax.scatter(y_test, y_pred, alpha=0.6, s=12)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
ax.plot(lims, lims, "k--", linewidth=1)
ax.set_xlabel("Actual solubility")
ax.set_ylabel("Predicted solubility")
ax.set_title(f"{MODEL_NAME}: Predicted vs Actual (R2={r2:.3f}, RMSE={rmse:.3f})")
ax.set_xlim(lims)
ax.set_ylim(lims)

out_path = out_dir / "predicted_vs_actual.png"
fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
plt.show()
out_path
